# Stage 5: Deep Learning — CNN & ResNet

- Inspect CNN and ResNet architectures
- Launch training with progress bars
- Plot training curves

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import json

from models.cnn    import AudioCNN
from models.resnet import AudioResNet
from features      import load_feature_cache
from train         import train_dl, set_seed
from config        import MODELS_DIR, PLOTS_DIR

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#0d0d1a',
    'text.color': 'white', 'axes.labelcolor': '#aaa',
    'xtick.color': '#aaa', 'ytick.color': '#aaa',
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1 · Architecture Summaries

In [ ]:
cnn    = AudioCNN()
resnet = AudioResNet()

dummy = torch.zeros(4, 1, 128, 173)
with torch.no_grad():
    out_cnn    = cnn(dummy)
    out_resnet = resnet(dummy)

cnn_params    = sum(p.numel() for p in cnn.parameters())
resnet_params = sum(p.numel() for p in resnet.parameters())

print(f'AudioCNN    output: {out_cnn.shape}    params: {cnn_params:,}')
print(f'AudioResNet output: {out_resnet.shape}  params: {resnet_params:,}')

## 2 · Load Features

In [ ]:
X, y, folds = load_feature_cache('logmel')
print(f'X: {X.shape}  y: {y.shape}  folds: {np.unique(folds)}')

## 3 · Train CNN

In [ ]:
set_seed()
cnn_model   = AudioCNN()
cnn_history = train_dl(cnn_model, 'cnn', X, y, folds,
                       epochs=40, lr=1e-3, batch_size=32)
print('CNN training done!')

## 4 · Train ResNet

In [ ]:
set_seed()
resnet_model   = AudioResNet(pretrained=False)
resnet_history = train_dl(resnet_model, 'resnet', X, y, folds,
                          epochs=40, lr=1e-3, batch_size=32)
print('ResNet training done!')

## 5 · Training Curves

In [ ]:
def plot_history(hist, name):
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, hist['train_loss'], color='#00d4ff', lw=2, label='Train')
    ax1.plot(epochs, hist['val_loss'],   color='#ff6b6b', lw=2, label='Val')
    ax1.set_title('Loss', color='white', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
    
    ax2.plot(epochs, [a*100 for a in hist['train_acc']], color='#00d4ff', lw=2, label='Train')
    ax2.plot(epochs, [a*100 for a in hist['val_acc']],   color='#ff6b6b', lw=2, label='Val')
    ax2.set_title('Accuracy (%)', color='white', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
    
    plt.suptitle(f'{name} Training Curves', fontsize=14, color='white', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../outputs/plots/training_history_{name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(cnn_history,    'CNN')
plot_history(resnet_history, 'ResNet')